# Tuần 03 — Thực hành kNN: 4 bài

Demo `01_Demo-kNN.ipynb` dừng ở việc **vote nhãn cho một mẫu test**.

Notebook này có **4 bài**. Mỗi bài là một bộ dữ liệu trên Kaggle. Cả 4 bài đều làm.

Viết hàm kNN một lần ở phần chung (NumPy, không dùng `KNeighborsClassifier`), rồi áp vào từng bài. Điền các ô có `???`. Giữ `np.random.seed(3000)`.

Tải đủ 4 file, đặt vào thư mục `data/`:

| Bài | Bộ dữ liệu | Link | File |
|---|---|---|---|
| 1 | Iris Species | https://www.kaggle.com/datasets/uciml/iris | `data/Iris.csv` |
| 2 | Palmer Penguins | https://www.kaggle.com/datasets/parulpandey/palmer-archipelago-antarctica-penguin-data | `data/penguins_size.csv` |
| 3 | Heart Failure Prediction | https://www.kaggle.com/datasets/fedesoriano/heart-failure-prediction | `data/heart.csv` |
| 4 | Mushroom Classification | https://www.kaggle.com/datasets/uciml/mushroom-classification | `data/mushrooms.csv` |


In [ ]:
import numpy as np
from matplotlib import pyplot as plt
from sklearn.preprocessing import LabelEncoder

np.set_printoptions(precision=3, suppress=True)


## Phần chung — hàm kNN

Các bài bên dưới gọi lại 4 hàm này.

**Chuẩn hóa min–max**, min/max tính trên từng cột của `x_train`, rồi áp cùng hai mốc đó cho `x_test`. Cột nào `max == min` thì giữ nguyên.

$$
x' = \frac{x - x_{min}}{x_{max} - x_{min}}
$$

**Khoảng cách Euclid** từ một mẫu test tới mọi dòng train:

$$
d_i = \sqrt{\sum_j (test_j - train_{i,j})^2}
$$

**Một mẫu:** lấy `K` chỉ số gần nhất (`np.argsort`), vote nhãn xuất hiện nhiều nhất. Hòa thì `np.argmax` trên `counts` chọn nhãn có mã nhỏ hơn.

**Cả tập test:** gọi dự đoán từng mẫu, trả về mảng nhãn cùng độ dài `x_test`.


In [ ]:
def chuan_hoa_minmax(x_train, x_test):
    x_min = x_train.min(axis=0)
    x_max = x_train.max(axis=0)
    diff = x_max - x_min
    diff = np.where(diff == 0, 1, diff)
    x_train_scaled = (x_train - x_min) / diff
    x_test_scaled = (x_test - x_min) / diff
    return x_train_scaled, x_test_scaled

def tinh_khoang_cach(test_s, x_train):
    return np.sqrt(np.sum((x_train - test_s) ** 2, axis=1))

def du_doan_1_mau(test_s, x_train, y_train, K):
    distances = tinh_khoang_cach(test_s, x_train)
    idx = np.argsort(distances)[:K]
    counts = np.bincount(y_train[idx])
    return np.argmax(counts)

def du_doan_tap(x_test, x_train, y_train, K):
    return np.array([du_doan_1_mau(x, x_train, y_train, K) for x in x_test])

In [ ]:
def chia_train_test(X, Y, ty_le=0.8, seed=3000):
    np.random.seed(seed)
    rand_indices = np.arange(len(Y))
    np.random.shuffle(rand_indices)
    n_train = int(ty_le * len(Y))
    train_idx = rand_indices[:n_train]
    test_idx = rand_indices[n_train:]
    return X[train_idx], Y[train_idx], X[test_idx], Y[test_idx]


def accuracy(y_pred, y_test):
    return np.mean(y_pred == y_test)


def ve_accuracy_theo_K(x_train, y_train, x_test, y_test, ds_K=None):
    if ds_K is None:
        ds_K = [1, 3, 5, 7, 9, 11, 13, 15]
    ds_acc = [accuracy(du_doan_tap(x_test, x_train, y_train, K), y_test) for K in ds_K]
    print(list(zip(ds_K, np.round(ds_acc, 3))))
    plt.figure(figsize=(6, 4))
    plt.plot(ds_K, ds_acc, marker='o')
    plt.xlabel('K')
    plt.ylabel('accuracy')
    plt.ylim(0, 1.05)
    plt.grid(True)
    plt.show()
    return ds_K, ds_acc


---

# Bài 1 — Iris

Nguồn: https://www.kaggle.com/datasets/uciml/iris

File `data/Iris.csv`. 150 bông, 3 loài.

Cột: `Id`, `SepalLengthCm`, `SepalWidthCm`, `PetalLengthCm`, `PetalWidthCm`, `Species`.

- Bỏ cột `Id`. Đó là số thứ tự, không phải số đo của hoa.
- Bốn cột đo là số (cm). Nhãn `Species` là chữ: dùng `LabelEncoder`.
- Chuẩn hóa min–max, rồi so sánh accuracy khi **có** và **không** chuẩn hóa (`K=5`).
- In kiểm tra mẫu test đầu tiên (khoảng cách K láng giềng, nhãn của họ, nhãn đoán, nhãn thật), rồi vẽ accuracy theo K.


In [ ]:
X_iris = raw_iris[:, 1:5].astype(float)
le = LabelEncoder()
Y_iris = le.fit_transform(raw_iris[:, 5])

In [ ]:
dist = tinh_khoang_cach(x_test_s[0], x_train_s)
idx = np.argsort(dist)[:K]
print('K khoảng cách nhỏ nhất:', dist[idx])
print('nhãn K láng giềng:', y_train[idx])
print('nhãn dự đoán:', du_doan_1_mau(x_test_s[0], x_train_s, y_train, K))
print('y_test[0]:', y_test[0])

print('accuracy K=5, đã chuẩn hóa:', accuracy(du_doan_tap(x_test_s, x_train_s, y_train, K), y_test))
print('accuracy K=5, chưa chuẩn hóa:', accuracy(du_doan_tap(x_test, x_train, y_train, K), y_test))

ve_accuracy_theo_K(x_train_s, y_train, x_test_s, y_test)

**Nộp kèm bài 1**

1. `X` có bao nhiêu dòng, bao nhiêu cột? Có bao nhiêu lớp?
2. Accuracy tại `K=5` khi có chuẩn hóa và khi không chuẩn hóa chênh nhau thế nào? Vì sao với Iris mức chênh thường nhỏ?
3. Vì sao không đưa cột `Id` vào feature?


In [ ]:
# 1. X có 150 dòng, 4 cột và 3 lớp.
# 2. Accuracy có chuẩn hóa và không chuẩn hóa thường rất gần nhau vì 4 đặc trưng đều cùng đơn vị cm và có thang đo tương đối gần nhau.
# 3. Id chỉ là số thứ tự, không mang thông tin hình thái của hoa nên đưa vào sẽ làm sai khoảng cách.

---

# Bài 2 — Palmer Penguins

Nguồn: https://www.kaggle.com/datasets/parulpandey/palmer-archipelago-antarctica-penguin-data

File `data/penguins_size.csv`. Đoán loài chim cánh cụt.

Cột: `species`, `island`, `culmen_length_mm`, `culmen_depth_mm`, `flipper_length_mm`, `body_mass_g`, `sex`.

- Bỏ mọi dòng có ô trống hoặc `NA`.
- Nhãn là `species`.
- `island` và `sex` là chữ: `LabelEncoder` **từng cột**.
- Bốn cột đo là số: `culmen_length_mm`, `culmen_depth_mm`, `flipper_length_mm`, `body_mass_g`.
- Bắt buộc chuẩn hóa. `body_mass_g` khoảng vài nghìn, chiều dài mỏ khoảng vài chục. Không chuẩn hóa thì khoảng cách gần như chỉ còn khối lượng.


In [ ]:
valid = ~np.any((raw_pen == '') | (raw_pen == 'NA'), axis=1)
clean_pen = raw_pen[valid]
le_island = LabelEncoder()
le_sex = LabelEncoder()
island = le_island.fit_transform(clean_pen[:, 1])
sex = le_sex.fit_transform(clean_pen[:, 6])
X_pen = np.column_stack([clean_pen[:, 2:6].astype(float), island, sex])
le_species = LabelEncoder()
Y_pen = le_species.fit_transform(clean_pen[:, 0])
print('số dòng bị bỏ:', len(raw_pen) - len(clean_pen))

In [ ]:
x_train, y_train, x_test, y_test = chia_train_test(X_pen, Y_pen)
x_train_s, x_test_s = chuan_hoa_minmax(x_train, x_test)

print('accuracy K=5, đã chuẩn hóa:', accuracy(du_doan_tap(x_test_s, x_train_s, y_train, 5), y_test))
print('accuracy K=5, chưa chuẩn hóa:', accuracy(du_doan_tap(x_test, x_train, y_train, 5), y_test))

ve_accuracy_theo_K(x_train_s, y_train, x_test_s, y_test)


**Nộp kèm bài 2**

1. Bỏ bao nhiêu dòng vì thiếu dữ liệu?
2. Accuracy `K=5` khi chưa chuẩn hóa thấp hơn bản đã chuẩn hóa ra sao? Cột nào kéo khoảng cách Euclid?
3. `K` nào cho accuracy cao nhất trên đồ thị?


In [ ]:
# 1. Số dòng bị bỏ = len(raw_pen) - len(clean_pen).
# 2. Chưa chuẩn hóa, body_mass_g có thang đo hàng nghìn nên chi phối khoảng cách Euclid mạnh hơn các cột còn lại.
# 3. K có accuracy cao nhất là K tương ứng với điểm cao nhất trên đồ thị.

---

# Bài 3 — Heart Failure Prediction

Nguồn: https://www.kaggle.com/datasets/fedesoriano/heart-failure-prediction

File `data/heart.csv`. 918 dòng. Đoán có bệnh tim hay không. Cùng kiểu bài với demo tình trạng xe.

Cột theo đúng thứ tự trong file:

`Age`, `Sex`, `ChestPainType`, `RestingBP`, `Cholesterol`, `FastingBS`, `RestingECG`, `MaxHR`, `ExerciseAngina`, `Oldpeak`, `ST_Slope`, `HeartDisease`.

- Nhãn `HeartDisease` đã là `0` / `1`.
- Cột chữ, `LabelEncoder` từng cột: `Sex`, `ChestPainType`, `RestingECG`, `ExerciseAngina`, `ST_Slope`.
- Cột số: `Age`, `RestingBP`, `Cholesterol`, `FastingBS`, `MaxHR`, `Oldpeak`.
- Ghép các cột đã mã hóa thành một ma trận `X` kiểu float, rồi chuẩn hóa min–max.
- Hai lớp, nên các `K` đem thử là số lẻ.


In [ ]:
enc_cols = [1, 2, 6, 8, 10]
X_heart = raw_heart[:, [0,1,2,3,4,5,6,7,8,9,10]].copy()
for j in range(X_heart.shape[1]):
    if j in [1,2,6,8,10]:
        X_heart[:,j] = LabelEncoder().fit_transform(X_heart[:,j])
X_heart = X_heart.astype(float)
Y_heart = raw_heart[:,11].astype(int)

In [ ]:
x_train, y_train, x_test, y_test = chia_train_test(X_heart, Y_heart)
x_train_s, x_test_s = chuan_hoa_minmax(x_train, x_test)

print('accuracy K=5:', accuracy(du_doan_tap(x_test_s, x_train_s, y_train, 5), y_test))
ve_accuracy_theo_K(x_train_s, y_train, x_test_s, y_test)


**Nộp kèm bài 3**

1. Có bao nhiêu người trong tập test? Tỉ lệ lớp `0` và `1` trên toàn bộ dữ liệu là bao nhiêu?
2. Accuracy cao nhất và `K` tương ứng?
3. Vì sao với bài 2 lớp nên chọn `K` lẻ?


In [ ]:
# 1. Tập test chiếm 20%: int(0.2 * 918) = 184 người. Tỉ lệ lớp 0/1 tính từ np.unique(Y_heart, return_counts=True).
# 2. Accuracy cao nhất và K tương ứng được lấy từ đồ thị/ve_accuracy_theo_K.
# 3. K lẻ giúp giảm khả năng hòa phiếu giữa hai lớp khi vote.

---

# Bài 4 — Mushroom Classification

Nguồn: https://www.kaggle.com/datasets/uciml/mushroom-classification

File `data/mushrooms.csv`. Đoán nấm ăn được (`e`) hay độc (`p`).

Mọi cột đều là chữ, cùng kiểu demo xe. Cột 0 là nhãn `class`. Các cột còn lại là feature.

- Bỏ dòng nào còn giá trị `?`.
- `LabelEncoder` **từng cột** feature, rồi mã hóa riêng cột nhãn. Giống đoạn mã hóa trong demo xe.
- **Không** chuẩn hóa min–max. Mã `LabelEncoder` chỉ là tên nhóm được đánh số, không phải số đo cùng đơn vị.
- Tập này vài nghìn dòng. `tinh_khoang_cach` cần tính một lần cho cả `x_train` (vector), không viết vòng `for` từng dòng train.


In [ ]:
valid = ~np.any(raw_mush == '?', axis=1)
clean_mush = raw_mush[valid]
X_mush = np.empty((len(clean_mush), clean_mush.shape[1]-1), dtype=float)
for j in range(1, clean_mush.shape[1]):
    X_mush[:, j-1] = LabelEncoder().fit_transform(clean_mush[:,j])
Y_mush = LabelEncoder().fit_transform(clean_mush[:,0])
print('số dòng còn lại:', len(clean_mush))

In [ ]:
x_train, y_train, x_test, y_test = chia_train_test(X_mush, Y_mush)

# Không gọi chuan_hoa_minmax
print('accuracy K=5:', accuracy(du_doan_tap(x_test, x_train, y_train, 5), y_test))
ve_accuracy_theo_K(x_train, y_train, x_test, y_test)


# 1. Số dòng còn lại = len(clean_mush), số feature = X_mush.shape[1].
# 2. Accuracy cao nhất và K tương ứng được lấy từ đồ thị/ve_accuracy_theo_K.
# 3. Bài nấm giống demo xe ở bước mã hóa các cột chữ thành số rồi áp dụng kNN; khác Iris/Penguins/Heart ở chỗ các feature sau mã hóa có cùng dạng mã số phân loại và notebook không yêu cầu chuẩn hóa chúng.

In [ ]:
# Bài 4
# 1.
# 2.
# 3.
